In [ ]:
import torch
print("CUDA Available: ", torch.cuda.is_available())
print("CUDA Device Name: ", torch.cuda.get_device_name(0))
torch.cuda.empty_cache()

# Verify CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, SparseVectorParams
from app.utils.settings import QDRANT_HOST, QDRANT_PORT, COLLECTION_NAME, CHUNKS_FILE
from app.utils.chunking import load_chunks

In [ ]:
def ingest_chunks_to_qdrant():
    """
    Ingests pre-chunked documents into Qdrant vector store using local embeddings.
    Creates the collection if it doesn't exist and adds texts + metadata in batch.
    """
    # 1. Initialize local embeddings model
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},  # change to "cuda" if GPU is available
        encode_kwargs={"normalize_embeddings": True},
    )

    # 2. Connect to Qdrant instance
    client = QdrantClient(
        host=QDRANT_HOST,
        port=QDRANT_PORT,
        timeout=120,
    )

    # 3. Create collection if it doesn't exist (fixed dimension + named vectors)
    if not client.collection_exists(collection_name=COLLECTION_NAME):
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config={
                "text-dense": VectorParams(size=384, distance=Distance.COSINE)
            },
            sparse_vectors_config={
                "text-sparse": SparseVectorParams()  # no size needed for sparse vectors
            },
        )
        print(f"Collection '{COLLECTION_NAME}' created with 'text-dense' vector.")
    else:
        print(f"Collection '{COLLECTION_NAME}' already exists.")

    # 4. Initialize LangChain Qdrant vector store with named vector
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
        vector_name="text-dense",          # must match the named vector created
        sparse_vector_name="text-sparse",  # optional, only if you plan to use hybrid search
        # force_recreate=True, # recreate collection (use with caution!)
    )

    # 5. Load chunks and prepare data for ingestion
    chunks = load_chunks(CHUNKS_FILE)
    if not chunks:
        print("No chunks found to ingest.")
        return

    texts = [chunk["content"] for chunk in chunks]
    metadatas = [
        {
            "release": chunk.get("release", ""),
            "series": chunk.get("series", ""),
            "spec": chunk.get("spec", ""),
            "text": chunk["content"]  # optional: store full text in payload if needed
        }
        for chunk in chunks
    ]

    # 6. Batch ingest texts + metadata into Qdrant (efficient and clean)
    vector_store.add_texts(
        texts=texts,
        metadatas=metadatas,
        # ids=[...]  # optional: provide custom IDs if you need deterministic behavior
    )

    print(f"Successfully ingested {len(texts)} chunks into Qdrant collection '{COLLECTION_NAME}'.")

In [ ]:
ingest_chunks_to_qdrant()

# Test collection

In [ ]:
client = QdrantClient(
        host=QDRANT_HOST,
        port=QDRANT_PORT,
        timeout=120
        )
collection_info = client.get_collection(COLLECTION_NAME)
print(collection_info)
print(f"\nCollection: {COLLECTION_NAME}")
print(f"Status: {collection_info.status}")
print(f"Points count: {collection_info.points_count:,}")